# Shoaling: several simulated fish in one dish

Every fish runs the trained model unchanged and sees the others as moving dots: do they shoal? F (with and without
the group repulsion term of 08_second_data_set.ipynb, and with a collision rule) and D4 trained on the screen (with
and without the dot's direction of motion), in the training dish and in one five times wider, where the fish start
together. The simulations and readouts are cached in `output/multi_fish/`:

    uv run python -m twin.multi_fish --checks          # the multi-fish simulator reduces to the standard one
    uv run python -m twin.multi_fish --run-grid        # pairs, groups of 5 and 10, both dishes, both seeds
    uv run python -m twin.multi_fish --analyse-grid    # -> summary_grid.json
    uv run python -m twin.multi_fish --run-pairdm      # pairs of the screen-trained D4, with and without direction
    uv run python -m twin.multi_fish --analyse-pairdm  # -> summary_pairdm.json
    uv run python -m twin.shoal_videos                 # the videos below

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Video, display
from twin import multi_fish as MF
from twin import paths
S = json.load(open(paths.MULTI_FISH / "summary_grid.json"))
P = json.load(open(paths.MULTI_FISH / "summary_pairdm.json"))


def two(f, a, b, nd=2):
    return f"{f(a):.{nd}f} | {f(b):.{nd}f}"


def blind_within_2mm(cond):
    """Share of time the blind fish of a condition (same fish, the others not shown) have a neighbour within 2 mm."""
    spec = MF.CONDS[cond]
    TB, _ = MF.tracks(MF.load(MF.blind_of(cond)), spec["N"])
    TB = TB[:spec["G"]]
    D = np.linalg.norm(TB[:, :, None] - TB[:, None, :], axis=-1)
    D[:, np.arange(spec["N"]), np.arange(spec["N"])] = np.inf
    return float(np.mean(D.min(2) <= 2))

## Groups of five (Table tab:shoal-grid)
Dispersion D (blind fish: -0.05 to 0), alignment index (five random headings: 0.40) and the share of time with the
nearest neighbour within 2 mm, seed 0 | seed 101; blind in brackets. Real groups of five (21 dpf, 12.6 cm arena):
D -0.48, alignment 0.46 (random 0.40).

In [2]:
rows = {}
for dish, A in (("training", 1), ("5x", 5)):
    for var, lab in (("F", "F"), ("grT", "+ repulsion"), ("grTC", "+ collision")):
        a, b = S[f"grid_A{A}_N5_{var}"], S[f"grid_A{A}_N5_{var}_s101"]
        r = {"D": two(lambda s: s["D"]["mean"], a, b), "alignment": two(lambda s: s["pol"]["mean"], a, b),
             "within 2 mm (%)": two(lambda s: 100 * s["nn_le2"], a, b, 1)}
        if var == "F":
            r["alignment"] += f" ({a['pol_blind']['mean']:.2f})"
            r["within 2 mm (%)"] += (f" ({100 * blind_within_2mm(f'grid_A{A}_N5_F'):.1f} | "
                                     f"{100 * blind_within_2mm(f'grid_A{A}_N5_F_s101'):.1f})")
        rows[(dish, lab)] = r
pd.DataFrame(rows).T

D           alignment        within 2 mm (%)
training F            -0.30 | -0.30  0.42 | 0.42 (0.40)  4.1 | 4.2 (1.3 | 1.2)
         + repulsion  -0.29 | -0.26         0.42 | 0.42              3.5 | 3.4
         + collision  -0.28 | -0.24         0.42 | 0.42              0.0 | 0.0
5x       F            -0.68 | -0.60  0.42 | 0.42 (0.40)  2.8 | 2.8 (0.3 | 0.2)
         + repulsion  -0.68 | -0.58         0.43 | 0.42              2.3 | 2.2
         + collision  -0.64 | -0.59         0.42 | 0.42              0.0 | 0.0

## Pairs (Table tab:shoal-pairs)
Training dish, 50 pairs per condition, seed 0 | seed 101: attraction index (blind F: about 0), median distance
between the two fish, alignment above pseudo-pairs (fish from different dishes; two random headings give 0.64).
Real pairs: attraction 0.56 +/- 0.08, distance 5-20 mm.

In [3]:
R = {"F": (P["grid_A1_N2_F"], P["grid_A1_N2_F_s101"]),
     "D4, screen": (P["pairdm_A1_sD4_s0"], P["pairdm_A1_sD4_s101"]),
     "+ direction": (P["pairdm_A1_sD4dm_s0"], P["pairdm_A1_sD4dm_s101"])}
pd.DataFrame({lab: {"attraction": two(lambda s: s["ai"]["mean"], *v),
                    "distance (mm)": two(lambda s: s["nn_median_mm"], *v, 1),
                    "alignment above pseudo-pairs": two(lambda s: s["pol"]["mean"] - s["pol_pseudo"], *v, 3)}
              for lab, v in R.items()}).T

,attraction,distance (mm),alignment above pseudo-pairs
F,0.35 | 0.32,14.9 | 15.0,0.016 | 0.019
"D4, screen",0.21 | 0.24,17.6 | 17.9,0.009 | 0.009
+ direction,0.26 | 0.24,17.0 | 18.3,0.036 | 0.034


## Videos
Seed 0. Each fish is drawn with its recent track; open the notebook locally to play them (`output/multi_fish/videos/`).

In [4]:
V = paths.MULTI_FISH / "videos" / "report"


def rel(p):
    """The video's path relative to this notebook's folder (an absolute path if it is on another drive)."""
    try:
        return Path(os.path.relpath(p)).as_posix()
    except ValueError:
        return str(p)


for name, title in (("F_A1_N2", "F, pairs, training dish"), ("F_A1_N5", "F, 5 fish, training dish"),
                    ("F_A1_N10", "F, 10 fish, training dish"), ("F_A5_N2", "F, pairs, 5x dish"),
                    ("F_A5_N5", "F, 5 fish, 5x dish"), ("F_A5_N10", "F, 10 fish, 5x dish"),
                    ("rep_A1_N2", "F with and without the repulsion term, pairs"),
                    ("rep_A1_N5", "F with and without the repulsion term, 5 fish"),
                    ("pairs_A1", "pairs of F, D4 screen and D4 + direction, training dish"),
                    ("pairs_A5", "the same, 5x dish"),
                    ("pairsm_A1", "speed-matched pairs, training dish"), ("pairsm_A5", "speed-matched pairs, 5x dish")):
    print(title)
    display(Video(rel(V / f"{name}.mp4"), embed=False, width=640, html_attributes="controls loop muted"))

F, pairs, training dish


F, 5 fish, training dish


F, 10 fish, training dish


F, pairs, 5x dish


F, 5 fish, 5x dish


...

F with and without the repulsion term, 5 fish


pairs of F, D4 screen and D4 + direction, training dish


the same, 5x dish


speed-matched pairs, training dish


speed-matched pairs, 5x dish
